[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Winfredy/SadTalker/blob/main/quick_demo.ipynb)

### SadTalker：Learning Realistic 3D Motion Coefficients for Stylized Audio-Driven Single Image Talking Face Animation

[arxiv](https://arxiv.org/abs/2211.12194) | [project](https://sadtalker.github.io) | [Github](https://github.com/Winfredy/SadTalker)

Wenxuan Zhang, Xiaodong Cun, Xuan Wang, Yong Zhang, Xi Shen, Yu Guo, Ying Shan, Fei Wang.

Xi'an Jiaotong University, Tencent AI Lab, Ant Group

CVPR 2023

TL;DR: A realistic and stylized talking head video generation method from a single image and audio


Installation (around 5 mins)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
IMAGE_PATH = "/content/drive/MyDrive/face.png"
AUDIO_PATH = "/content/drive/MyDrive/viseme_master.wav"

print("Resim:", IMAGE_PATH)
print("Ses  :", AUDIO_PATH)

Resim: /content/drive/MyDrive/face.png
Ses  : /content/drive/MyDrive/viseme_master.wav


In [5]:
import os

print(os.path.exists("/content/drive/MyDrive/viseme_master.wav"))

True


In [29]:
!ls -lah "/content/drive/MyDrive/viseme_master.wav"

ls: cannot access '/content/drive/MyDrive/viseme_master.wav': No such file or directory


In [30]:
!find /content/drive/MyDrive -iname "*viseme*"

^C


In [7]:
### make sure that CUDA is available in Edit -> Nootbook settings -> GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

NVIDIA L4, 23034 MiB, 22564 MiB


In [8]:
import os

print("SadTalker patch uygulanıyor...")

# --------------------------------------------------
# 1) torchvision.functional_tensor -> functional
# --------------------------------------------------

file1 = "/usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py"

if os.path.exists(file1):
    with open(file1, "r", encoding="utf-8") as f:
        txt = f.read()

    txt = txt.replace(
        "from torchvision.transforms.functional_tensor import rgb_to_grayscale",
        "from torchvision.transforms.functional import rgb_to_grayscale"
    )

    with open(file1, "w", encoding="utf-8") as f:
        f.write(txt)

    print("✓ basicsr torchvision patch")


# --------------------------------------------------
# 2) np.float -> float
# --------------------------------------------------

file2 = "/content/SadTalker/src/face3d/util/my_awing_arch.py"

if os.path.exists(file2):
    with open(file2, "r", encoding="utf-8") as f:
        txt = f.read()

    txt = txt.replace("np.float", "float")

    with open(file2, "w", encoding="utf-8") as f:
        f.write(txt)

    print("✓ np.float patch")


# --------------------------------------------------
# 3) preprocess.py trans_params düzeltmesi
# --------------------------------------------------

file3 = "/content/SadTalker/src/face3d/util/preprocess.py"

if os.path.exists(file3):
    with open(file3, "r", encoding="utf-8") as f:
        txt = f.read()

    old = "trans_params = np.array([w0, h0, s, t[0], t[1]])"

    new = """trans_params = np.array([
    float(w0),
    float(h0),
    float(s),
    float(np.squeeze(t)[0]),
    float(np.squeeze(t)[1])
])"""

    txt = txt.replace(old, new)

    with open(file3, "w", encoding="utf-8") as f:
        f.write(txt)

    print("✓ trans_params patch")


# --------------------------------------------------
# 4) NumPy int/float alias düzeltmesi
# --------------------------------------------------

for fpath in [
    "/content/SadTalker/src/face3d/util/preprocess.py",
    "/content/SadTalker/src/face3d/util/my_awing_arch.py"
]:
    if os.path.exists(fpath):
        with open(fpath, "r", encoding="utf-8") as f:
            txt = f.read()

        txt = txt.replace("np.int", "int")
        txt = txt.replace("np.bool", "bool")

        with open(fpath, "w", encoding="utf-8") as f:
            f.write(txt)

print("✓ numpy alias patch")


# --------------------------------------------------
# 5) Kurulum kontrolü
# --------------------------------------------------

try:
    import torch
    print("Torch:", torch.__version__)
    print("CUDA :", torch.cuda.is_available())
except Exception as e:
    print("Torch hata:", e)

try:
    import basicsr
    print("✓ basicsr")
except Exception as e:
    print("basicsr hata:", e)

try:
    import facexlib
    print("✓ facexlib")
except Exception as e:
    print("facexlib hata:", e)

try:
    import gfpgan
    print("✓ gfpgan")
except Exception as e:
    print("gfpgan hata:", e)

print("\nPatch tamamlandı.")

SadTalker patch uygulanıyor...
✓ basicsr torchvision patch
✓ np.float patch
✓ trans_params patch
✓ numpy alias patch
Torch: 2.11.0+cu128
CUDA : True
✓ basicsr
✓ facexlib
✓ gfpgan

Patch tamamlandı.


In [9]:
print('Download pre-trained models...')
!rm -rf checkpoints
!bash scripts/download_models.sh

Download pre-trained models...
bash: scripts/download_models.sh: No such file or directory


In [10]:
!update-alternatives --install /usr/local/bin/python3 python3 /usr/bin/python3.8 2
!update-alternatives --install /usr/local/bin/python3 python3 /usr/bin/python3.9 1
!sudo apt install python3.8

!sudo apt-get install python3.8-distutils

!python --version

!apt-get update

!apt install software-properties-common

!sudo dpkg --remove --force-remove-reinstreq python3-pip python3-setuptools python3-wheel

!apt-get install python3-pip

print('Git clone project and install requirements...')
!git clone https://github.com/Winfredy/SadTalker &> /dev/null
%cd SadTalker
!export PYTHONPATH=/content/SadTalker:$PYTHONPATH
!python3.8 -m pip install torch==1.12.1+cu113 torchvision==0.13.1+cu113 torchaudio==0.12.1 --extra-index-url https://download.pytorch.org/whl/cu113
!apt update
!apt install ffmpeg &> /dev/null
!python3.8 -m pip install -r requirements.txt

update-alternatives: error: alternative path /usr/bin/python3.9 doesn't exist
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
python3.8 is already the newest version (3.8.20-1+jammy1).
0 upgraded, 0 newly installed, 0 to remove and 63 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
python3.8-distutils is already the newest version (3.8.20-1+jammy1).
0 upgraded, 0 newly installed, 0 to remove and 63 not upgraded.
Python 3.12.13
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jamm

In [11]:
!sed -i 's/warnings.filterwarnings("ignore", category=np.VisibleDeprecationWarning)/warnings.filterwarnings("ignore")/g' \
/content/SadTalker/SadTalker/src/face3d/util/preprocess.py

In [12]:
# Torch ile uyumlu setuptools
!pip install -q "setuptools<82" "pip<26"

# Ana paketler
!pip install -q \
numpy==1.26.4 \
scipy==1.11.4 \
imageio \
imageio-ffmpeg \
librosa==0.10.1 \
resampy==0.4.3 \
face_alignment==1.3.5 \
kornia==0.7.3 \
yacs==0.1.8 \
joblib==1.4.2 \
scikit-image==0.22.0 \
opencv-python \
av \
safetensors \
filterpy \
einops \
tqdm

# SadTalker bağımlıları
!pip install -q basicsr==1.4.2
!pip install -q facexlib==0.3.0
!pip install -q gfpgan==1.3.8

In [14]:
!which python3
!ls -l /usr/bin/python*
!ls -l /usr/local/bin/python*

/usr/local/bin/python3
-rwxr-xr-x 1 root root 5937704 Mar  3 11:56 /usr/bin/python3.10
lrwxrwxrwx 1 root root      34 Mar  3 11:56 /usr/bin/python3.10-config -> x86_64-linux-gnu-python3.10-config
-rwxr-xr-x 1 root root 7905264 Mar  4 09:23 /usr/bin/python3.12
lrwxrwxrwx 1 root root      34 Mar  4 09:23 /usr/bin/python3.12-config -> x86_64-linux-gnu-python3.12-config
-rwxr-xr-x 1 root root 5146792 Sep  7  2024 /usr/bin/python3.8
lrwxrwxrwx 1 root root      17 Aug  8  2024 /usr/bin/python3-config -> python3.10-config
-r-xr-xr-x 1 root root 155 Jan  1  2000 /usr/local/bin/python
lrwxrwxrwx 1 root root  25 May 26 13:13 /usr/local/bin/python3 -> /etc/alternatives/python3


In [15]:
!sudo ln -sf /usr/bin/python3.12 /usr/bin/python3

In [16]:
!ls -l /usr/bin/python3
!python3 --version

lrwxrwxrwx 1 root root 19 Jun  4 13:45 /usr/bin/python3 -> /usr/bin/python3.12
Python 3.12.13


In [19]:
print("Kullanılan resim:", IMAGE_PATH)
print("Kullanılan ses  :", AUDIO_PATH)

!python inference.py \
  --source_image "$IMAGE_PATH" \
  --driven_audio "$AUDIO_PATH" \
  --result_dir results \
  --size 512 \
  --preprocess full \
  --still \
  --enhancer gfpgan

Kullanılan resim: /content/drive/MyDrive/face.png
Kullanılan ses  : /content/drive/MyDrive/viseme_master.wav
Traceback (most recent call last):
  File "/content/SadTalker/inference.py", line 3, in <module>
    import torch
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 2229, in <module>
    from torch import _VF as _VF, functional as functional  # usort: skip
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/functional.py", line 8, in <module>
    import torch.nn.functional as F
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/__init__.py", line 8, in <module>
    from torch.nn.modules import *  # usort: skip # noqa: F403
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/__init__.py", line 2, in <module>
    from .linear import Bilinear, Identity, LazyLinear, Linear  # usort: skip
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [13]:
!grep -n "functional_tensor" \
/usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py

In [14]:
file_path = "/usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py"

with open(file_path, "r", encoding="utf-8") as f:
    txt = f.read()

txt = txt.replace(
    "from torchvision.transforms.functional_tensor import rgb_to_grayscale",
    "from torchvision.transforms.functional import rgb_to_grayscale"
)

with open(file_path, "w", encoding="utf-8") as f:
    f.write(txt)

print("Patch uygulandı")

Patch uygulandı


In [15]:
!grep -n "rgb_to_grayscale" \
/usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py

8:from torchvision.transforms.functional import rgb_to_grayscale
631:        img_gray = rgb_to_grayscale(img, num_output_channels=1)


In [16]:
file_path = "/content/SadTalker/src/face3d/util/preprocess.py"

with open(file_path, "r", encoding="utf-8") as f:
    txt = f.read()

txt = txt.replace(".astype(int32)", ".astype(np.int32)")

with open(file_path, "w", encoding="utf-8") as f:
    f.write(txt)

print("int32 patch tamam")

int32 patch tamam


Download models (1 mins)